In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')
print(os.getcwd())

C:\Users\Klara\retail-intelligence


In [2]:
import pandas as pd

fold_results = pd.read_csv('data/processed/walkforward_folds.csv')
fold_results_ok = fold_results[fold_results['DataQualityFlag'] == 'OK'].copy()

print(f"Products: {fold_results_ok['StockCode'].nunique()}")
print(f"Folds: {len(fold_results_ok)}")

Products: 18
Folds: 90


In [3]:
worst_mape = fold_results_ok.nlargest(5, 'Prophet_MAPE')[
    ['StockCode', 'Fold', 'Prophet_MAE', 'Prophet_MAPE', 'MeanActualDemand']
]
print(worst_mape.to_string(index=False))

StockCode  Fold  Prophet_MAE  Prophet_MAPE  MeanActualDemand
    21915     5   536.918512   1166.597229            301.25
    15036     2   616.055398   1027.561852             75.00
    15036     3   350.024397    662.655060            261.00
    15036     1   596.986919    568.593218            264.00
    21977     4   480.022832    535.776914            102.25


# Walk-Forward Validation — Deep-Dive Observations

## Why is mean MAPE (136.0%) much higher than median MAPE (73.4%)?

The five worst Prophet MAPE folds were investigated directly, revealing two separate causes.

### 1. Low-demand weeks inflate MAPE mechanically

- Product **15036** appears in **3 of the 5** worst folds.
- All three folds have mean actual demand below **265 units**.
- Although the 5-unit minimum-demand guard prevents true near-zero explosions, moderate demand levels can still produce large percentage errors from relatively small absolute mistakes.

This reinforces an earlier finding rather than contradicting it: **15036** was already identified as a consistent underperformer against the naive baseline.

### 2. Genuine forecasting failures

Not all extreme MAPE values are denominator artifacts.

For example:

- Product **21915** (fold 5)
- MAE ≈ **537 units**
- Mean actual demand ≈ **301 units**

The forecasting error was nearly twice the actual demand, representing a genuine forecasting miss.

### Conclusion

Mean MAPE is inflated by both low-volume products and genuine forecasting failures, while median MAPE is more robust to extreme values.

For this reason, both metrics are reported instead of relying on a single summary statistic.

---

# Walk-Forward Validation — Questions and Answers

## Q: Why use walk-forward validation instead of a simple train/test split?

**Answer:**

A single train/test split evaluates performance on only one specific time window and may produce conclusions that do not generalize to future periods.

Walk-forward validation better reflects real-world forecasting because the model is repeatedly retrained as new data becomes available and tested on the next four weeks. By rolling forward five times, it produces a distribution of results rather than a single performance number.

This approach also revealed that product **21977**, which appeared problematic in Week 4's single holdout test, actually beats the naive baseline in **3 out of 5 folds (60%)**, showing that its earlier underperformance was likely due to one particular test window rather than a persistent issue.

---

## Q: What does a 47.8% win rate against the naive baseline actually mean?

**Answer:**

Across all **90 evaluated folds** (18 products × 5 folds, after excluding two low-reliability products), Prophet outperformed the naive baseline in **47.8%** of cases.

This means that the simple strategy of repeating last week's demand slightly outperformed Prophet overall. Rather than being a weakness of the analysis, this is an important finding: naive forecasting is a surprisingly strong baseline for short-term retail demand.

The aggregate result also hides substantial variation between products:

- **15036** and **22086** lose to naive in every fold.
- **84946**, **22469**, **21212**, **85123A**, **85099B**, and **22616** outperform naive in **4 out of 5 folds**.

---

## Q: What does the 70% win rate against seasonal naive mean, and why is it different from the naive result?

**Answer:**

The seasonal naive model predicts demand using the same period from the previous year. Prophet outperforms this baseline in **70%** of folds, indicating a consistent advantage.

However, the dataset contains only about one year of observations. Because seasonal naive relies on repeating yearly patterns, it has limited information from which to learn genuine seasonality.

This result suggests that Prophet captures trend information more effectively than seasonal naive captures seasonality in this dataset.

---

## Q: Why does MAPE vary so much across products?

**Answer:**

Two separate factors explain the variation.

**First**, low-demand products naturally produce inflated percentage errors because even moderate forecasting mistakes become large relative to actual demand.

- Product **15036** accounts for **3 of the 5** highest-MAPE folds.
- All of these folds have mean actual demand below **265 units**.

**Second**, some products genuinely experience large forecasting failures.

- Product **21915** (fold 5) produced an absolute error nearly twice as large as the actual demand.

These two effects explain why mean MAPE (**136.0%**) is much larger than median MAPE (**73.4%**).

---

## Q: What would be done differently with more time?

**Answer:**

### 1. Product-specific tuning

Products **15036** and **22086** lose to the naive baseline in all five folds.

This pattern is consistent enough to justify testing:

- Higher or lower `changepoint_prior_scale`
- Logistic growth with a cap
- Product-specific Prophet configurations

Importantly, these changes should be tested only for these products rather than applied globally.

### 2. More historical data

Several findings in this project are influenced by the limited history available.

For example:

- Yearly seasonality was disabled.
- Seasonal naive underperformed Prophet.

Additional historical data would allow stronger conclusions about which seasonal patterns represent true signal and which are simply noise.

---

# Key Takeaways

- Prophet beats the naive baseline in **47.8%** of folds.
- Prophet beats the seasonal naive baseline in **70%** of folds.
- Performance varies significantly across products and time periods.
- Products **15036** and **22086** consistently underperform against naive.
- Mean MAPE (**136.0%**) is heavily influenced by extreme cases, while median MAPE (**73.4%**) is more robust.
- Walk-forward validation reveals patterns that a single train/test split would miss.